# Spatial Join


In the previous section, we looked at how to determine spatial relationships between features using spatial predicates. Now let's turn to an operation that puts those relationships to work — **combining attribute data from two layers based on their spatial positions relative to one another**.

A **spatial join** is a method that links the attribute information of two spatial datasets according to their geometric relationship.

With a spatial join, you can, for example:

- determine which district each point belongs to;
- aggregate data by spatial unit.

In GeoPandas, this is done using the `sjoin` function, which lets you specify both the spatial predicate and the type of join.

In this section, we will walk through how to perform spatial joins and use them to analyse geodata.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import pandas as pd
import geopandas as gpd

### 0.2. Preparing the Data


This section uses files from our [repository](https://github.com/bella-mir/geoPythonEn/tree/main/data/spb):

- **spb_admin.gpkg** — polygon data on the boundaries of Saint Petersburg's districts and municipal units.  
  Source: course materials for "Methods of Spatial Analysis", HSE University (R. Goncharov)

- **spb_theaters.csv** — data on theatres in Saint Petersburg.  
  Source: [Saint Petersburg Open Data Portal](https://data.gov.spb.ru/)


Let's read the municipal unit boundaries for Saint Petersburg from the GeoPackage file.


In [ ]:
okrug = gpd.read_file("../../data/spb/spb_admin.gpkg", layer="okrug")

okrug.explore(tiles="cartodbpositron")

Let's read the theatre data from the CSV file and create a `GeoDataFrame` from it.


In [ ]:
theaters_csv = pd.read_csv("../../data/spb/spb_theaters.csv", index_col=0)
theaters_csv = theaters_csv.dropna(subset=["longitude", "latitude"])

theaters_gdf = gpd.GeoDataFrame(
    theaters_csv,
    geometry=gpd.points_from_xy(theaters_csv["longitude"], theaters_csv["latitude"]),
    crs="EPSG:4326"
)

theaters_gdf.explore(tiles="cartodbpositron")

## 1. Joining the Data


Let's determine which municipal unit each theatre is located in. To do this, we will perform a spatial join between the theatres layer and the municipal units layer.

Recall that a spatial join relies on a spatial predicate. Here, we need to find **which unit each theatre falls inside**, so we will use the `within` predicate.

Before running the join, it is important to make sure both layers are in the same CRS. If they differ, one of the layers must be reprojected first.


### 1.1. Checking the CRS


Let's check whether the two layers share the same CRS:


In [ ]:
okrug.crs == theaters_gdf.crs

They match, so we can proceed without reprojecting.


### 1.2. Performing the Spatial Join


For each theatre, we want to identify the municipal unit it falls within. We use `sjoin` with the `within` predicate and a left join (`how="left"`) to retain all theatres, including any that do not fall within a unit boundary.


In [ ]:
theaters_in_okrug = gpd.sjoin(
    theaters_gdf,
    okrug,
    how="left",
    predicate="within"
)

The spatial join appends the attributes of the municipal unit to each theatre feature, based on which unit it falls within. Theatres that fall outside every unit keep the theatre attributes and get `NaN` in the joined columns — let's check whether there are any:

In [ ]:
theaters_in_okrug["NAME"].isna().sum()

Every theatre fell inside a municipal unit, so nothing was lost.

Let's inspect the result. We'll display theatre names alongside their corresponding unit names, using the field names as they appear in the source data: `name` for theatres and `NAME` for municipal units.

In [ ]:
theaters_in_okrug[["name", "NAME"]].head()

We now have information on which municipal unit each theatre is located in.


## 2. Aggregating the Results


Now that the spatial join has assigned a municipal unit to each theatre, we can aggregate the results — for example, to count how many theatres are located in each unit.

Let's group the data by unit name (`NAME`) and count the number of theatres in each. The result is a table showing the theatre count per municipal unit, sorted in descending order.


In [ ]:
theater_counts = (
    theaters_in_okrug
    .groupby("NAME")
    .size()
    .reset_index(name="theater_count")
    .sort_values("theater_count", ascending=False)
)

theater_counts.head()

In this way, a spatial join allows us to move from analysing individual features to analysing spatial units.


## Summary


In this section, we covered the spatial join — a method for linking data from two layers based on their spatial relationship.

We saw how the `sjoin` function can be used to match features across layers using a spatial predicate.

We also demonstrated how the result of a spatial join can feed directly into further analysis — such as aggregating data and counting features within defined spatial units.
